**Preprocessing for the UNSW-NB15 Dataset**

This notebook turns the decisions from the EDA (Section 11 of `eda.ipynb`) into the data both of us train on. Since we split the work (T0/T1 and T2/T3), we need to use **exactly the same** train/validation/test data and preprocessing, otherwise our results can't be compared.

It produces two things:
- `data/processed/train.csv`, `val.csv` and `test.csv`: the splits, saved so we load the same rows every time
- `src/preprocessing.py`: the shared `build_preprocessor()` that turns the raw columns into 73 numeric features

Since `data/` is not on GitHub, each of us has to run this notebook once. With the fixed `RANDOM_STATE` we get the same splits.

**1. Setup**

In [1]:
import sys
sys.path.append("..") # so we can import from src/

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.preprocessing import RANDOM_STATE, NOT_FEATURES, build_preprocessor

DATA = Path("../data")
OUT = DATA / "processed"
OUT.mkdir(exist_ok=True)

**2. Load the data and remove duplicates**

Same as in the EDA: we remove rows that are identical in every column except `id`. We only do this for the training file, the test file stays as it is since real traffic also repeats.

In [2]:
train_raw = pd.read_csv(DATA / "UNSW_NB15_training_set.csv")
test_raw  = pd.read_csv(DATA / "UNSW_NB15_testing_set.csv")

train_dedup = train_raw.drop_duplicates(subset=[c for c in train_raw.columns if c != "id"])
print("training file:", len(train_raw), "---> after removing duplicates:", len(train_dedup))
print("test file:", len(test_raw))

training file: 175341 ---> after removing duplicates: 107740
test file: 82332


**3. Sample 30,000 rows and split into train (24,000) and validation (6,000)**

We stratify on `attack_cat` instead of `label`. This keeps the attack/normal balance the same in every split (since `attack_cat` already tells us the label), and also makes sure rare attack types like Worms show up in both train and validation.

**Why do we do this?**: 72 models, some with thousands of polynomial columns, have to train in reasonable time on a laptop. 24,000 training rows is still far more than enough ---> computational constraint

In [3]:
sample, _ = train_test_split(train_dedup, train_size=30_000, stratify=train_dedup["attack_cat"], random_state=RANDOM_STATE)
train, val = train_test_split(sample, test_size=6_000, stratify=sample["attack_cat"], random_state=RANDOM_STATE)
test = test_raw

print("train:", train.shape, " val:", val.shape, " test:", test.shape)

train: (24000, 45)  val: (6000, 45)  test: (82332, 45)


In [4]:
split_check = pd.DataFrame({
    "train": train["label"].value_counts(normalize=True),
    "val": val["label"].value_counts(normalize=True),
    "test": test["label"].value_counts(normalize=True),
}).rename(index={0: "Normal", 1: "Attack"})
split_check.round(3)

,train,val,test
label,,,
Attack,0.518,0.518,0.551
Normal,0.482,0.482,0.449


In [5]:
pd.DataFrame({
    "train": train["attack_cat"].value_counts(),
    "val": val["attack_cat"].value_counts(),
    "test": test["attack_cat"].value_counts(),
})

,train,val,test
attack_cat,,,
Analysis,355,89,677
Backdoor,342,85,583
DoS,848,212,4089
Exploits,4421,1105,11132
Fuzzers,3598,899,6062
Generic,931,233,18871
Normal,11559,2890,37000
Reconnaissance,1675,419,3496
Shellcode,243,61,378


Train and validation have the same balance (around 52% attack), while the test file has a bit more attacks (55.1%). The test file also comes from a different capture, so we expect some gap between validation and test performance later.

**4. The shared preprocessor**

`build_preprocessor()` in `src/preprocessing.py` applies all the EDA decisions in one sklearn `ColumnTransformer`:

| Columns | Steps | Output columns |
|---|---|---|
| `proto`, `service`, `state` | top 15 `proto` and rare `state` ---> "other", then one-hot | 35 |
| 31 skewed numeric (`LOG_COLS`) | `log1p`, then `StandardScaler` | 31 |
| 7 other numeric (`SCALE_COLS`) | `StandardScaler` | 7 |
| `id`, `label`, `attack_cat`, `is_ftp_login` | dropped | 0 |

The categories are fixed lists from the EDA, so a value never seen in training (the test file has 5 rows with `state` = `ACC` or `CLO`) just gets all zeros for `state` instead of crashing.

**Why do we do this?**: The scaler learns the mean and std from the data, so it has to be fit on the 24,000 training rows only and then reused on validation and test ---> avoid leakage. Putting it in a pipeline makes this automatic.

In [6]:
X_train = train.drop(columns=NOT_FEATURES)
X_val = val.drop(columns=NOT_FEATURES)
X_test = test.drop(columns=NOT_FEATURES)

prep = build_preprocessor()
Xt_train = prep.fit_transform(X_train) # fit on train only
Xt_val = prep.transform(X_val)
Xt_test = prep.transform(X_test)

print("train:", Xt_train.shape, " val:", Xt_val.shape, " test:", Xt_test.shape)
print("missing values:", np.isnan(Xt_train).sum(), np.isnan(Xt_val).sum(), np.isnan(Xt_test).sum())

train: (24000, 73)  val: (6000, 73)  test: (82332, 73)
missing values: 0 0 0


Quick check that it did what we expect: the scaled columns of the training set should have mean 0 and std 1, while validation and test are only close to that (they were scaled with the training mean and std).

In [7]:
names = prep.get_feature_names_out()
is_scaled = ~pd.Series(names).str.startswith("cat__")

for split, Xt in [("train", Xt_train), ("val", Xt_val), ("test", Xt_test)]:
    m, s = Xt[:, is_scaled].mean(axis=0), Xt[:, is_scaled].std(axis=0)
    print(f"{split:>5}: mean of column means = {m.mean():.3f}, mean of column stds = {s.mean():.3f}")

train: mean of column means = 0.000, mean of column stds = 1.000
  val: mean of column means = 0.003, mean of column stds = 0.993
 test: mean of column means = -0.051, mean of column stds = 1.112


Validation is very close to train, which is expected since both come from the same file. The test file is a bit further off (std around 1.11 instead of 1) ---> this is expected for the test file

In [8]:
pd.Series(names).str.split("__").str[0].value_counts() # number of columns from each part

cat    35
log    31
num     7
Name: count, dtype: int64

**5. Save the splits**

We save the raw columns (not the transformed ones), because each model fits its own pipeline: preprocessor ---> transformation ---> model.

In [9]:
train.to_csv(OUT / "train.csv", index=False)
val.to_csv(OUT / "val.csv", index=False)
test.to_csv(OUT / "test.csv", index=False)
print([p.name for p in OUT.glob("*.csv")])

['val.csv', 'test.csv', 'train.csv']
